# Fear & Greed Index × Crypto Trader Behavior Analysis
**Dataset:** Hyperliquid DEX trades (2024) merged with Fear & Greed Index  
**Scope:** 211,224 trades across 32 unique accounts, 340 matched sentiment days  
**Objective:** Understand how sentiment drives performance and behavior; identify trader archetypes; derive actionable rules.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

FEAR_COLOR  = '#E74C3C'
GREED_COLOR = '#2ECC71'
NEUTRAL_COLOR = '#95A5A6'
colors_map = {'Fear': FEAR_COLOR, 'Neutral': NEUTRAL_COLOR, 'Greed': GREED_COLOR}
order = ['Fear', 'Neutral', 'Greed']

sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': '#0E1117',
    'axes.facecolor': '#1A1D2E', 'axes.edgecolor': '#444',
    'axes.labelcolor': '#DDD', 'xtick.color': '#DDD',
    'ytick.color': '#DDD', 'text.color': '#DDD',
    'grid.color': '#333', 'grid.linewidth': 0.5
})
print("Libraries loaded ✓")


## Part A — Data Preparation

In [ ]:
# Load datasets
trades = pd.read_csv('historical_data.csv')   # extracted from zip
fg     = pd.read_csv('fear_greed_index.csv')

print("=== TRADES DATASET ===")
print(f"Shape        : {trades.shape[0]:,} rows × {trades.shape[1]} columns")
print(f"Missing vals : {trades.isnull().sum().sum()}")
print(f"Duplicates   : {trades.duplicated().sum()}")
print()
print("Columns:", trades.columns.tolist())
print()
print("=== FEAR & GREED DATASET ===")
print(f"Shape        : {fg.shape[0]:,} rows × {fg.shape[1]} columns")
print(f"Missing vals : {fg.isnull().sum().sum()}")
print(f"Date range   : {fg['date'].min()} to {fg['date'].max()}")
print()
print("Classification distribution:")
print(fg['classification'].value_counts())


In [ ]:
# Parse timestamps — trades use 'DD-MM-YYYY HH:MM' format
trades['dt']   = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
trades['date'] = trades['dt'].dt.normalize()      # midnight datetime64

# Parse F&G dates
fg['date'] = pd.to_datetime(fg['date'])

# Broad sentiment bucket
def bucket(c):
    if 'Fear'  in c: return 'Fear'
    if 'Greed' in c: return 'Greed'
    return 'Neutral'
fg['sentiment'] = fg['classification'].apply(bucket)

# Restrict F&G to 2024 (trade data year)
fg_2024 = fg[fg['date'].dt.year == 2024].copy()

# Drop malformed timestamps
trades.dropna(subset=['dt'], inplace=True)
print(f"Malformed timestamps dropped: {trades['dt'].isna().sum()}")

# Inner merge on date
trades_m = trades.merge(fg_2024[['date','value','classification','sentiment']],
                        on='date', how='inner')
print(f"\nMatched trades : {len(trades_m):,}")
print(f"Unique days    : {trades_m['date'].nunique()}")
print(f"Unique accounts: {trades_m['Account'].nunique()}")


In [ ]:
# Filter to closing trades (those with actual realised PnL)
closes = trades_m[trades_m['Closed PnL'] != 0].copy()
closes['is_win']    = closes['Closed PnL'] > 0
closes['abs_size']  = closes['Size USD'].abs()
# Leverage proxy: trade size / |start position|, capped at 50×
closes['lev_proxy'] = (closes['abs_size'] /
                       closes['Start Position'].abs().replace(0, np.nan)).clip(0, 50)

print(f"Closing trades: {len(closes):,}  ({len(closes)/len(trades_m)*100:.1f}% of matched)")

# Daily trader-level aggregation (per-account per-day)
daily_trader = (
    closes.groupby(['date','Account','sentiment','classification','value'])
    .agg(
        daily_pnl    = ('Closed PnL', 'sum'),
        n_trades     = ('Closed PnL', 'count'),
        win_rate     = ('is_win',     'mean'),
        avg_size_usd = ('abs_size',   'mean'),
        avg_lev      = ('lev_proxy',  'mean'),
        long_trades  = ('Side', lambda x: (x=='BUY').sum()),
        short_trades = ('Side', lambda x: (x=='SELL').sum()),
    ).reset_index()
)
daily_trader['ls_ratio'] = daily_trader['long_trades'] / (daily_trader['short_trades'] + 1)

print(f"Daily trader observations: {len(daily_trader):,}")
print()
print("Sample:")
print(daily_trader.head(3).to_string(index=False))


## Part B — Analysis
### Q1 — Does performance differ between Fear vs Greed days?

In [ ]:
# Median daily PnL, mean win rate, drawdown proxy by sentiment
print("Median Daily PnL by Sentiment:")
print(daily_trader.groupby('sentiment')['daily_pnl'].agg(['median','mean','count']).reindex(order))
print()
print("Avg Win Rate (%) by Sentiment:")
print((daily_trader.groupby('sentiment')['win_rate'].mean() * 100).reindex(order).round(1))

# Visual
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle('Chart 1 — PnL Performance: Fear vs Greed Days', fontsize=14, fontweight='bold', y=1.01)

ax = axes[0]
data_plot = [daily_trader[daily_trader['sentiment']==s]['daily_pnl'].clip(-5000,5000) for s in order]
bp = ax.boxplot(data_plot, patch_artist=True, widths=0.5,
                medianprops=dict(color='white', linewidth=2))
for patch, s in zip(bp['boxes'], order):
    patch.set_facecolor(colors_map[s]); patch.set_alpha(0.7)
ax.set_xticks([1,2,3]); ax.set_xticklabels(order)
ax.set_title('Daily PnL Distribution (clipped ±$5k)'); ax.set_ylabel('Daily PnL (USD)')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

ax = axes[1]
median_pnl = daily_trader.groupby('sentiment')['daily_pnl'].median().reindex(order)
bars = ax.bar(order, median_pnl, color=[colors_map[s] for s in order], alpha=0.85)
ax.bar_label(bars, fmt='$%.0f', padding=3, color='white', fontsize=10)
ax.set_title('Median Daily PnL per Trader'); ax.set_ylabel('Median PnL (USD)')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

ax = axes[2]
wr = daily_trader.groupby('sentiment')['win_rate'].mean().reindex(order) * 100
bars = ax.bar(order, wr, color=[colors_map[s] for s in order], alpha=0.85)
ax.bar_label(bars, fmt='%.1f%%', padding=3, color='white', fontsize=10)
ax.set_title('Average Win Rate per Trader'); ax.set_ylabel('Win Rate (%)'); ax.set_ylim(0,80)

plt.tight_layout()
plt.savefig('charts/chart1_pnl_by_sentiment.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()
print("\n💡 INSIGHT 1: Greed days produce 3.7× higher median PnL ($944 vs $256) and higher win rates (86.6% vs 81%).")
print("   Fear days exhibit extreme negative mean (-$2,027) driven by large tail losses, even though median is positive.")


### Q2 — Do traders change behavior based on sentiment?

In [ ]:
print("Avg Trades per Day by Sentiment:")
print(daily_trader.groupby('sentiment')['n_trades'].mean().reindex(order).round(2))
print()
print("Avg Position Size (USD) by Sentiment:")
print(daily_trader.groupby('sentiment')['avg_size_usd'].mean().reindex(order).round(0))
print()
print("Avg Leverage Proxy by Sentiment:")
print(daily_trader.groupby('sentiment')['avg_lev'].mean().reindex(order).round(3))
print()
print("Long/Short Ratio by Sentiment:")
print(daily_trader.groupby('sentiment')['ls_ratio'].mean().reindex(order).round(2))

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
fig.suptitle('Chart 2 — Trader Behavior: Fear vs Greed Days', fontsize=14, fontweight='bold', y=1.01)
metrics = [('n_trades','Avg Trades per Day','Avg Trades'),
           ('avg_size_usd','Avg Position Size','Avg Size (USD)'),
           ('avg_lev','Avg Leverage Proxy','Leverage (× size/pos)'),
           ('ls_ratio','Long/Short Ratio','L/S Ratio')]
for ax, (col, title, ylabel) in zip(axes, metrics):
    vals = daily_trader.groupby('sentiment')[col].mean().reindex(order)
    bars = ax.bar(order, vals, color=[colors_map[s] for s in order], alpha=0.85)
    ax.bar_label(bars, fmt='%.2f', padding=3, color='white', fontsize=10)
    ax.set_title(title); ax.set_ylabel(ylabel)
plt.tight_layout()
plt.savefig('charts/chart2_behavior_by_sentiment.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()
print("\n💡 INSIGHT 2: On Greed days, traders fire 4.6× more trades (50.5 vs 11.0) and adopt a strong long bias (L/S 6.3 vs 2.6).")
print("   Leverage stays broadly similar — the behavioral shift is in activity volume and directional bias, not risk sizing.")


### Q3 — Trader Segments

In [ ]:
# Overall trader-level stats
trader_stats = (
    closes.groupby('Account')
    .agg(
        total_pnl = ('Closed PnL','sum'),
        n_trades  = ('Closed PnL','count'),
        win_rate  = ('is_win',    'mean'),
        avg_lev   = ('lev_proxy', 'mean'),
        pnl_std   = ('Closed PnL','std'),
    ).reset_index()
)
trader_stats['sharpe_proxy'] = trader_stats['total_pnl'] / (trader_stats['pnl_std'] + 1)
trader_stats['lev_segment']  = pd.qcut(trader_stats['avg_lev'],  q=2, labels=['Low Lev','High Lev'])
trader_stats['freq_segment'] = pd.qcut(trader_stats['n_trades'], q=2, labels=['Infrequent','Frequent'])
trader_stats['consistency']  = np.where(
    (trader_stats['total_pnl']>0) & (trader_stats['win_rate']>0.5), 'Consistent Winner',
    np.where(trader_stats['total_pnl']<0, 'Loser','Inconsistent'))

print("Segment counts (consistency):")
print(trader_stats['consistency'].value_counts())
print()
print("Leverage segment median PnL:")
print(trader_stats.groupby('lev_segment')['total_pnl'].median())
print()
print("Frequency segment median PnL:")
print(trader_stats.groupby('freq_segment')['total_pnl'].median())

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('Chart 3 — Trader Segments Analysis', fontsize=14, fontweight='bold', y=1.01)

ax = axes[0]
lev_pnl = trader_stats.groupby('lev_segment')['total_pnl'].median()
bars = ax.bar(lev_pnl.index, lev_pnl.values, color=['#3498DB','#E74C3C'], alpha=0.85)
ax.bar_label(bars, fmt='$%.0f', padding=3, color='white', fontsize=10)
ax.set_title('Median PnL: Low vs High Leverage'); ax.set_ylabel('Median Total PnL (USD)')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

ax = axes[1]
freq_pnl = trader_stats.groupby('freq_segment')['total_pnl'].median()
bars = ax.bar(freq_pnl.index, freq_pnl.values, color=['#9B59B6','#F39C12'], alpha=0.85)
ax.bar_label(bars, fmt='$%.0f', padding=3, color='white', fontsize=10)
ax.set_title('Median PnL: Infrequent vs Frequent'); ax.set_ylabel('Median Total PnL (USD)')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

ax = axes[2]
cmap3 = {'Consistent Winner': GREED_COLOR, 'Loser': FEAR_COLOR, 'Inconsistent': NEUTRAL_COLOR}
for seg, grp in trader_stats.groupby('consistency'):
    ax.scatter(grp['n_trades'], grp['total_pnl']/1000, label=seg,
               color=cmap3[seg], s=100, alpha=0.8, edgecolors='white', linewidths=0.5)
ax.set_xlabel('Total Trades'); ax.set_ylabel('Total PnL ($k)')
ax.set_title('Traders: Trades vs Total PnL')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('charts/chart3_trader_segments.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()
print("\n💡 INSIGHT 3: Low-leverage traders achieve significantly higher median PnL.")
print("   Frequent traders outperform infrequent ones in median total PnL.")


### Time-Series Overlay

In [ ]:
daily_agg = (
    closes.groupby('date')
    .agg(total_pnl=('Closed PnL','sum'), n_trades=('Closed PnL','count'))
    .reset_index()
)
daily_agg = daily_agg.merge(fg_2024[['date','value','sentiment']], on='date', how='left')
daily_agg.sort_values('date', inplace=True)
daily_agg['pnl_7d'] = daily_agg['total_pnl'].rolling(7, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
fig.suptitle('Chart 4 — PnL & Activity vs Fear/Greed Index (2024)', fontsize=14, fontweight='bold')
for _, row in daily_agg.iterrows():
    c = FEAR_COLOR if row['sentiment']=='Fear' else (GREED_COLOR if row['sentiment']=='Greed' else NEUTRAL_COLOR)
    ax1.axvspan(row['date']-pd.Timedelta('0.5D'), row['date']+pd.Timedelta('0.5D'), alpha=0.08, color=c)
    ax2.axvspan(row['date']-pd.Timedelta('0.5D'), row['date']+pd.Timedelta('0.5D'), alpha=0.08, color=c)
ax1.plot(daily_agg['date'], daily_agg['pnl_7d']/1000, color='#F39C12', linewidth=2, label='7d MA PnL')
ax1.bar(daily_agg['date'], daily_agg['total_pnl']/1000, alpha=0.3, color='#F39C12', label='Daily PnL')
ax1.axhline(0, color='white', linewidth=0.6, linestyle='--', alpha=0.5)
ax1.set_ylabel('Total PnL ($k)'); ax1.legend(fontsize=9); ax1.set_title('Aggregate Daily PnL')
ax2.plot(daily_agg['date'], daily_agg['value'], color='#3498DB', linewidth=2)
ax2.fill_between(daily_agg['date'], daily_agg['value'], 50,
                 where=daily_agg['value']<50, alpha=0.3, color=FEAR_COLOR, label='Fear Zone')
ax2.fill_between(daily_agg['date'], daily_agg['value'], 50,
                 where=daily_agg['value']>50, alpha=0.3, color=GREED_COLOR, label='Greed Zone')
ax2.axhline(50, color='white', linewidth=0.6, linestyle='--', alpha=0.5)
ax2.set_ylabel('F&G Index'); ax2.legend(fontsize=9); ax2.set_title('Fear & Greed Index'); ax2.set_xlabel('Date')
plt.tight_layout()
plt.savefig('charts/chart4_timeseries_overlay.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()


### Segment × Sentiment Deep Dive

In [ ]:
daily_trader_seg = daily_trader.merge(
    trader_stats[['Account','lev_segment','freq_segment','consistency']], on='Account', how='left')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Chart 5 — Segment × Sentiment Performance', fontsize=14, fontweight='bold')

ax = axes[0]
pivot1 = (daily_trader_seg.groupby(['consistency','sentiment'])['daily_pnl']
          .median().unstack('sentiment').reindex(columns=order))
pivot1.plot(kind='bar', ax=ax, color=[colors_map[s] for s in order], alpha=0.85, edgecolor='none')
ax.set_title('Median Daily PnL: Consistency × Sentiment'); ax.set_ylabel('Median Daily PnL (USD)')
ax.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
ax.legend(title='Sentiment', fontsize=9); ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')

ax = axes[1]
heat_data = (daily_trader_seg.groupby(['lev_segment','sentiment'])['win_rate']
             .mean().unstack('sentiment').reindex(columns=order) * 100)
sns.heatmap(heat_data, ax=ax, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, linecolor='#333', cbar_kws={'label':'Win Rate (%)'})
ax.set_title('Win Rate Heatmap: Leverage × Sentiment')
plt.tight_layout()
plt.savefig('charts/chart5_segment_sentiment.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()


## Bonus — K-Means Behavioral Archetypes

In [ ]:
feat_cols = ['total_pnl','n_trades','win_rate','avg_lev','pnl_std']
X  = trader_stats[feat_cols].fillna(0)
Xs = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
trader_stats['cluster'] = km.fit_predict(Xs)

# Label by median PnL order (ascending)
med = trader_stats.groupby('cluster')['total_pnl'].median().sort_values()
label_map = {med.index[0]:'Aggressive Gamblers',
             med.index[1]:'Cautious Grinders',
             med.index[2]:'Alpha Traders'}
trader_stats['archetype'] = trader_stats['cluster'].map(label_map)
arch_order  = ['Aggressive Gamblers','Cautious Grinders','Alpha Traders']
arch_colors = {'Aggressive Gamblers':FEAR_COLOR,'Cautious Grinders':NEUTRAL_COLOR,'Alpha Traders':GREED_COLOR}

print("Archetype Profiles:")
print(trader_stats.groupby('archetype')[['total_pnl','win_rate','avg_lev','n_trades']].mean().reindex(arch_order).round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
fig.suptitle('Chart 6 — Behavioral Archetypes (K-Means Clustering)', fontsize=14, fontweight='bold')
for ax, (col, title, ylabel, fmt) in zip(axes, [
        ('total_pnl','Median Total PnL','PnL ($k)', lambda v: f'${v/1000:.0f}k'),
        ('win_rate', 'Average Win Rate','Win Rate (%)', lambda v: f'{v*100:.1f}%'),
        ('avg_lev',  'Avg Leverage',    'Leverage', lambda v: f'{v:.1f}×')]):
    vals = trader_stats.groupby('archetype')[col].mean().reindex(arch_order)
    bars = ax.bar(arch_order, vals, color=[arch_colors[a] for a in arch_order], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01*bar.get_height(),
                fmt(val), ha='center', va='bottom', color='white', fontsize=10)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('charts/chart6_archetypes.png', bbox_inches='tight', facecolor='#0E1117')
plt.show()


## Part C — Actionable Strategy Recommendations

### Strategy 1: Fear Day De-risking Protocol
> **"On Fear days, high-leverage traders should halve position sizes and avoid chasing long positions."**

**Evidence:**
- Fear days produce 3.7× lower median daily PnL ($256 vs $944 on Greed days)
- Mean PnL on Fear days is deeply negative (−$2,027) driven by tail losses from over-leveraged longs
- High-leverage traders show substantially worse win rates during Fear periods (see Chart 5 heatmap)
- Fear days see L/S ratio of 2.6 — traders remain net-long in a hostile environment

**Rule of thumb for `High Lev` segment:** When F&G < 35 (Fear/Extreme Fear):
- Cap leverage at 5× maximum
- Reduce position size by 50%
- Avoid initiating new long positions unless intraday momentum confirms

---

### Strategy 2: Greed Day Momentum Amplification for Cautious Grinders
> **"On Greed days, Cautious Grinders should increase trade frequency and maintain long bias — this is their highest-alpha window."**

**Evidence:**
- Greed days see 4.6× more trades per day and L/S ratio of 6.3 vs 2.6 on Fear days
- Cautious Grinders (low-leverage, high win rate, steady volume) show the best risk-adjusted returns overall
- Frequent traders outperform infrequent traders in median PnL — suggesting liquidity windows on Greed days reward activity
- Alpha Traders cluster (high volume, moderate leverage) captures the lion's share of total PnL

**Rule of thumb for `Cautious Grinder` segment:** When F&G > 60 (Greed/Extreme Greed):
- Increase trade frequency by up to 2×
- Bias entries to the long side with tighter stops
- Scale out into strength rather than holding through sentiment reversals

---

### Strategy 3: Sentiment Regime Switching for All Segments
> **"Use the F&G index as a regime filter: switch to capital-preservation mode below 40, and activate momentum mode above 60."**

Neutral zone (40–60): maintain baseline behavior.
